# 05 — Analyses added in the revision

The reviewers of the ML4H 2026 submission asked for a set of analyses that the
first version did not have. This notebook runs all of them, in order, from what
`03_train` and `04_results` left on disk. Each step is a script in
`src/revision/`, so it can also be run on its own (`python src/revision/<name>.py`
from the repository root); the scripts carry the detailed documentation.

**Inputs.** `data/train/predictions/`, `data/train/models/` and
`data/train/tables/` (from `03` and `04`), and `data/master/`.
**Outputs.** CSV and parquet files in `data/train/revision/`, read by `06_paper_outputs`.

**Time convention.** A forecast computed from row *t* (cases up to *t-1*,
weather up to *t*) is issued at the start of month *t+2*, when the cases of *t-1*
have had about 60 days to consolidate. Horizon *h* = 1 is then a nowcast of a
month that has already ended, *h* = 2 forecasts the month that begins on the
issue date, and *h* = 3 is the first horizon with a full month of warning. The
effective lead time is *h* − 2 months.

**Two reference months for "calm".** Onset and continuation are defined by the
municipality's outbreak state in a reference month: *t*, the month the row reads
(the reference of the submitted version, not yet consolidated on the issue date),
or *t-1*, the last month that is consolidated on the issue date (real time).

**Uncertainty.** 95% intervals come from a bootstrap over municipalities (1,000
replicates, fixed seed), which captures variation between municipalities. The
range across test years is reported beside it, because it is usually wider.

| Step | Script | Retrains | Approx. time |
|---|---|---|---|
| Checks | `comum.checar_metricas`, `treino.conferir_reproducao` | one fold | 2 min |
| A3 | `a3_tempo_real.py`: onsets that started before the issue date; which inputs are published by then | no | 1 min |
| C1 | `c1_onset_continuacao.py`: onset and continuation, h = 1 to 6, both references, with CIs | no | 2 min |
| A5 | `a5_incerteza.py`: Tables 1 and 2 with CIs | no | 1 min |
| A6 | `a6_orcamento_calmos.py`: fixed alert budget among calm municipalities, and the alerts-versus-onsets curve | no | 3 min |
| A9 | `a9_shap_onset.py`: SHAP by feature family on onset rows | no | 5 min |
| B1 | `b1_dobra_2025.py`: 2025 as a fourth test year, h = 1 to 6 | 2025 fold | 15 min |
| B2 | `b2_modelo_onset.py`: a model trained only on calm months | yes | 10 min |
| B3 | `b3_limiares_passado.py`: relative-definition thresholds from past years only | yes | 15 min |
| B4 | `b4_tuning.py`: random hyperparameter search, chosen on the year before each test year | yes | 45 min |
| C2 | `c2_calibracao.py`: isotonic calibration fitted on the year before each test year | 2021 fold | 5 min |

In [ ]:
import runpy
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
REVISION = PROJECT_ROOT / "src" / "revision"
sys.path.insert(0, str(REVISION))
OUT = PROJECT_ROOT / "data" / "train" / "revision"

def run(script):
    """Run one analysis script as if from the command line, and time it."""
    t0 = time.time()
    runpy.run_path(str(REVISION / script), run_name="__main__")
    print(f"\n[{script}: {time.time() - t0:.0f}s]")

## Checks before anything else

The weighted metrics used for the bootstrap must match scikit-learn, including
when the highest-scored row belongs to a municipality a replicate did not draw.
Retraining the 2024 fold of dengue/inc100/h = 1 must reproduce the predictions
that `03` saved, which is what lets the retraining analyses below reuse the
saved 2022 to 2024 predictions as their reference.

In [ ]:
import comum
import treino

comum.checar_metricas()
print("weighted AUROC and AP match scikit-learn")
treino.conferir_reproducao();

## Part A — Evaluation by issue date, without retraining

**A3.** How many real-time onsets (calm at *t-1*, outbreak at *t+h*) had in fact
already begun before the issue date, and which inputs are not yet published on
that date.

In [ ]:
run("a3_tempo_real.py")

**C1.** Base rate, recall and precision at 0.5, AUROC and AUPRC, separately for
onsets and continuations, at every horizon and with both reference months.

In [ ]:
run("c1_onset_continuacao.py")

**A5.** The submitted version's Tables 1 and 2 (dengue, inc100, *h* = 1,
reference *t*) with 95% intervals. The cell after it checks that these
recomputations, which use their own code, agree with the tables `04` wrote.

In [ ]:
run("a5_incerteza.py")

In [ ]:
# Two independent code paths must give the same headline numbers.
t04 = pd.read_csv(PROJECT_ROOT / "data" / "train" / "tables" / "table3_global_dengue_inc100.csv", index_col=0)
a5 = pd.read_csv(OUT / "a5_tabela1_ic.csv").set_index("model")
for model, label in [("rf", "Random Forest"), ("xgb", "XGBoost"), ("lgbm", "LightGBM"),
                     ("logreg", "Logistic regression"), ("persistence", "Persistence (month t)"),
                     ("persistence_tm1", "Persistence (real time)"), ("climatology_corr", "Climatology")]:
    for metric in ("auroc", "auprc"):
        assert abs(round(a5.loc[model, metric], 3) - t04.loc[label, metric]) < 1e-9, (model, metric)
onset04 = pd.read_csv(PROJECT_ROOT / "data" / "train" / "tables" / "table_onset_vs_continuation.csv", index_col=0)
c1 = pd.read_csv(OUT / "c1_onset_continuacao.csv")
for model, label in [("rf", "Random Forest"), ("xgb", "XGBoost"), ("climatology_corr", "Climatology")]:
    for subset, col in (("onset", "recall_onset"), ("continuation", "recall_continuation")):
        v = c1[(c1.h == 1) & (c1.ref == "t") & (c1.subset == subset) & (c1.model == model)]["recall"].item()
        assert abs(v - onset04.loc[label, col]) < 1e-12, (model, subset)
print("04 and the revision scripts agree on Tables 1 and 2")

**A6.** Each month, rank only the municipalities that were calm in the last
consolidated month and support the top *k*, or a fixed share of them. The
proportional budgets, from 1% to 100% of calm municipalities, trace the
trade-off between alerts raised and onsets caught.

In [ ]:
run("a6_orcamento_calmos.py")

**A9.** SHAP attributions by feature family on onset rows, against the pooled
sample and continuation rows (random forest, dengue, inc100, *h* = 1).

In [ ]:
run("a9_shap_onset.py")

## Part B — Analyses that retrain models

All use the configuration of `03_train` (same features, hyperparameters and
expanding-window folds) unless the analysis is precisely about changing one of
them.

**B1.** 2025 as a fourth test year: trained on every target before 2025. The
2025 fold played no part in any design choice; the features, definitions,
hyperparameters and the 0.5 threshold were all fixed on 2022 to 2024.

In [ ]:
run("b1_dobra_2025.py")

**B2.** A model trained only on months in which the municipality was calm, compared
with the single model on the same calm test months.

In [ ]:
run("b2_modelo_onset.py")

**B3.** The two relative definitions (historical range and standard score) with
thresholds computed only from years before each test year, models retrained on
those labels.

In [ ]:
run("b3_limiares_passado.py")

**B4.** A random search of 20 configurations per model, chosen on the year before
each test year, so the test year plays no part in the choice.

In [ ]:
run("b4_tuning.py")

**C2.** Isotonic calibration fitted on the out-of-sample predictions of the year
before each test year, and what the calibrated scores mean for a rule that acts
when the probability exceeds the ratio of the cost of acting to the loss avoided.

In [ ]:
run("c2_calibracao.py")

In [ ]:
print("Files written to", OUT.relative_to(PROJECT_ROOT))
for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<40} {p.stat().st_size / 1024:8.0f} KB")